# TravelFit — Preprocessing Data Destinasi Wisata

Notebook ini menyiapkan data destinasi untuk dua kebutuhan berbeda:

1. **K-Means (offline):** harga, rating, fasilitas, dan kategori destinasi.
2. **AHP–TOPSIS (runtime):** matriks keputusan C1–C6 yang dihitung setelah profil survei pengguna tersedia.

Dataset utama: *Indonesia Tourism Destination* (`tourism_with_id.csv` dan `tourism_rating.csv`). Fasilitas dan aktivitas yang diturunkan dari deskripsi diberi label **heuristik** dan harus diverifikasi sebelum digunakan sebagai data produksi.

> **Google Colab:** notebook otomatis melakukan mount Google Drive dan menggunakan folder `/content/drive/MyDrive/TravelFit`. Dataset dan output akan tersimpan permanen di folder **Drive Saya → TravelFit**.

## Pemetaan data terhadap enam kriteria

| Kriteria | Jenis | Sumber destinasi | Sumber pengguna |
|---|---|---|---|
| C1 Harga tiket | Cost | `Price` | Budget maksimal |
| C2 Rating | Benefit | `Rating` | Tingkat kepentingan kualitas |
| C3 Jarak | Cost | `Lat`, `Long` | Koordinat/kota asal |
| C4 Fasilitas | Benefit | Fasilitas terverifikasi atau heuristik deskripsi | Fasilitas yang dibutuhkan |
| C5 Kategori | Benefit | `Category` | Kategori yang diminati |
| C6 Hobi | Benefit | Tag aktivitas | Hobi pengguna |

Budget dan wilayah digunakan sebagai **filter keras**. Kategori dan hobi tidak difilter keras karena keduanya tetap menjadi pembeda C5 dan C6.

In [1]:
from __future__ import annotations

import json
import math
import re
import sys
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
try:
    from IPython.display import display, Markdown
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)
    Markdown = str

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
RANDOM_STATE = 42
print('Library siap.')

Library siap.


In [ ]:
# Google Drive akan menjadi penyimpanan permanen ketika notebook dijalankan di Colab.
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/TravelFit')
else:
    # Fallback untuk eksekusi lokal dari root repository atau folder notebooks.
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name.lower() == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'preprocessing'
for directory in (RAW_DIR, PROCESSED_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DATA_URLS = {
    'destinations': 'https://raw.githubusercontent.com/akhiyarwaladi/indonesia_tourism/main/tourism_with_id.csv',
    'ratings': 'https://raw.githubusercontent.com/akhiyarwaladi/indonesia_tourism/main/tourism_rating.csv',
}
RAW_FILES = {
    'destinations': RAW_DIR / 'tourism_with_id.csv',
    'ratings': RAW_DIR / 'tourism_rating.csv',
}

print('Environment  :', 'Google Colab + Drive' if IN_COLAB else 'Local')
print('Project root :', PROJECT_ROOT)
print('Raw data     :', RAW_DIR)
print('Processed    :', PROCESSED_DIR)

In [ ]:
def download_if_missing(url: str, destination: Path) -> Path:
    """Unduh data hanya jika file lokal belum tersedia."""
    if destination.exists() and destination.stat().st_size > 0:
        print(f'[reuse] {destination.name}')
        return destination
    print(f'[download] {destination.name}')
    urlretrieve(url, destination)
    return destination

for key, url in DATA_URLS.items():
    download_if_missing(url, RAW_FILES[key])

destinations_raw = pd.read_csv(RAW_FILES['destinations'])
ratings_raw = pd.read_csv(RAW_FILES['ratings'])

print('Destinations:', destinations_raw.shape)
print('Ratings     :', ratings_raw.shape)

In [ ]:
display(destinations_raw.head(3))
display(ratings_raw.head(3))

In [ ]:
def table_audit(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    return pd.DataFrame({
        'table': table_name,
        'column': df.columns,
        'dtype': [str(df[c].dtype) for c in df.columns],
        'non_null': [int(df[c].notna().sum()) for c in df.columns],
        'missing': [int(df[c].isna().sum()) for c in df.columns],
        'missing_pct': [round(float(df[c].isna().mean() * 100), 2) for c in df.columns],
        'unique': [int(df[c].nunique(dropna=True)) for c in df.columns],
    })

raw_audit = pd.concat([
    table_audit(destinations_raw, 'tourism_with_id'),
    table_audit(ratings_raw, 'tourism_rating'),
], ignore_index=True)
display(raw_audit)

## 1. Standardisasi struktur dan tipe data

Tahap berikut tidak langsung menghapus baris bermasalah. Notebook membuat flag kualitas agar keputusan pembersihan dapat dilacak dan dijelaskan pada tahap Data Understanding.

In [ ]:
def snake_case(value: str) -> str:
    value = re.sub(r'[^0-9a-zA-Z]+', '_', str(value).strip()).strip('_')
    return value.lower()

def clean_text(value):
    if pd.isna(value):
        return pd.NA
    value = re.sub(r'\s+', ' ', str(value)).strip()
    return value if value else pd.NA

destinations = destinations_raw.copy()
destinations.columns = [snake_case(c) for c in destinations.columns]
destinations = destinations.loc[:, ~destinations.columns.str.startswith('unnamed')]

ratings = ratings_raw.copy()
ratings.columns = [snake_case(c) for c in ratings.columns]
ratings = ratings.loc[:, ~ratings.columns.str.startswith('unnamed')]

text_columns = ['place_name', 'description', 'category', 'city']
for column in text_columns:
    if column in destinations.columns:
        destinations[column] = destinations[column].map(clean_text).astype('string')

numeric_columns = ['place_id', 'price', 'rating', 'time_minutes', 'lat', 'long']
for column in numeric_columns:
    if column in destinations.columns:
        destinations[column] = pd.to_numeric(destinations[column], errors='coerce')

for column in ['user_id', 'place_id', 'place_ratings']:
    ratings[column] = pd.to_numeric(ratings[column], errors='coerce')

destinations['source_dataset'] = 'Indonesia Tourism Destination (Kaggle mirror)'
destinations['source_url'] = DATA_URLS['destinations']
destinations['last_verified_at'] = pd.NA
print(destinations.dtypes)

In [ ]:
# Flag validitas. Baris tidak langsung dibuang agar alasan eksklusi tetap dapat diaudit.
destinations['duplicate_place_id'] = destinations.duplicated('place_id', keep=False)
destinations['duplicate_name_coordinate'] = destinations.duplicated(
    ['place_name', 'lat', 'long'], keep=False
)
destinations['valid_price'] = destinations['price'].ge(0) & destinations['price'].notna()
destinations['valid_rating'] = destinations['rating'].between(1, 5, inclusive='both')
destinations['valid_coordinate'] = (
    destinations['lat'].between(-90, 90, inclusive='both')
    & destinations['long'].between(-180, 180, inclusive='both')
)
# Bounding box kasar Indonesia hanya menjadi warning, bukan aturan penghapusan otomatis.
destinations['inside_indonesia_bbox'] = (
    destinations['lat'].between(-11.5, 6.5, inclusive='both')
    & destinations['long'].between(94.0, 142.0, inclusive='both')
)
destinations['eligible_for_ranking'] = (
    destinations['place_id'].notna()
    & destinations['place_name'].notna()
    & destinations['valid_price']
    & destinations['valid_rating']
    & destinations['valid_coordinate']
    & ~destinations['duplicate_place_id']
)

quality_flags = [
    'duplicate_place_id', 'duplicate_name_coordinate', 'valid_price',
    'valid_rating', 'valid_coordinate', 'inside_indonesia_bbox',
    'eligible_for_ranking',
]
display(destinations[quality_flags].agg(['sum', 'mean']).T)

In [ ]:
# Kota pada dataset utama dipetakan ke provinsi untuk mendukung filter wilayah.
CITY_TO_PROVINCE = {
    'Jakarta': 'DKI Jakarta',
    'Bandung': 'Jawa Barat',
    'Semarang': 'Jawa Tengah',
    'Yogyakarta': 'DI Yogyakarta',
    'Surabaya': 'Jawa Timur',
}
destinations['province'] = destinations['city'].map(CITY_TO_PROVINCE).astype('string')

CATEGORY_MAP = {
    'budaya': 'budaya',
    'taman hiburan': 'hiburan',
    'cagar alam': 'alam',
    'pusat perbelanjaan': 'belanja',
    'tempat ibadah': 'religi',
    'bahari': 'bahari',
}
destinations['category_original'] = destinations['category']
destinations['category_clean'] = (
    destinations['category']
    .str.casefold()
    .map(CATEGORY_MAP)
    .fillna(destinations['category'].str.casefold().str.replace(r'\s+', '_', regex=True))
    .astype('string')
)
display(destinations[['category_original', 'category_clean']].drop_duplicates().sort_values('category_clean'))

## 2. Penanganan durasi dan agregasi rating

`Time_Minutes` bukan bagian dari C1–C6, tetapi dipertahankan sebagai informasi tambahan. Nilai kosong diisi dengan median kategori dan diberikan indikator missing agar prosesnya tidak tersembunyi. Rating utama C2 tetap menggunakan kolom `Rating`; rating pengguna hanya menjadi atribut pendukung dan bahan validasi.

In [ ]:
destinations['time_minutes_missing'] = destinations['time_minutes'].isna()
category_time_median = destinations.groupby('category_clean')['time_minutes'].transform('median')
global_time_median = destinations['time_minutes'].median()
destinations['time_minutes_imputed'] = (
    destinations['time_minutes']
    .fillna(category_time_median)
    .fillna(global_time_median)
)

ratings_valid = ratings.loc[
    ratings['place_id'].notna()
    & ratings['user_id'].notna()
    & ratings['place_ratings'].between(1, 5, inclusive='both')
].drop_duplicates(['user_id', 'place_id'], keep='last')

rating_summary = (
    ratings_valid.groupby('place_id', as_index=False)
    .agg(
        user_rating_mean=('place_ratings', 'mean'),
        user_rating_count=('place_ratings', 'count'),
        user_rating_std=('place_ratings', 'std'),
    )
)
rating_summary['user_rating_std'] = rating_summary['user_rating_std'].fillna(0)
destinations = destinations.merge(rating_summary, on='place_id', how='left', validate='one_to_one')
destinations['c1_ticket_price'] = destinations['price'].astype('Float64')
destinations['c2_rating'] = destinations['rating'].astype('Float64')

display(destinations[['place_name', 'rating', 'user_rating_mean', 'user_rating_count']].head())

## 3. Ekstraksi fasilitas C4

Dataset utama belum menyediakan fasilitas dalam kolom terstruktur. Notebook mendeteksi **penyebutan** fasilitas di dalam deskripsi. Tidak ditemukannya kata kunci berarti *tidak disebutkan*, bukan berarti fasilitas pasti tidak tersedia.

Untuk hasil akhir penelitian, kolom ini sebaiknya diverifikasi menggunakan SISPARNAS, OpenStreetMap, situs resmi, atau survei lapangan.

In [ ]:
FACILITY_KEYWORDS = {
    'toilet': ['toilet', 'kamar mandi', 'wc umum'],
    'parking': ['parkir', 'parking'],
    'food': ['warung', 'restoran', 'rumah makan', 'kafe', 'cafe', 'food court', 'kuliner'],
    'worship': ['mushola', 'musala', 'masjid', 'tempat ibadah', 'gereja', 'pura', 'vihara'],
    'accessibility': ['disabilitas', 'difabel', 'kursi roda', 'wheelchair', 'aksesibel'],
    'information_center': ['pusat informasi', 'information center', 'layanan informasi'],
    'lodging': ['penginapan', 'hotel', 'homestay', 'resort'],
    'guide': ['pemandu wisata', 'tour guide', 'pemandu lokal'],
}
CORE_FACILITIES = ['toilet', 'parking', 'food', 'worship', 'accessibility', 'information_center']

def normalize_for_search(value) -> str:
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value).casefold()).strip()

def contains_keyword(text: str, keyword: str) -> bool:
    """Cocokkan kata/frasa utuh agar 'selam' tidak cocok dengan 'selama'."""
    escaped = re.escape(keyword.casefold()).replace(r'\ ', r'\s+')
    pattern = rf'(?<!\w){escaped}(?!\w)'
    return re.search(pattern, text) is not None

description_search = destinations['description'].map(normalize_for_search)
for facility, keywords in FACILITY_KEYWORDS.items():
    destinations[f'facility_{facility}_mentioned'] = description_search.map(
        lambda text, keys=keywords: int(any(contains_keyword(text, key) for key in keys))
    )

facility_columns = [f'facility_{name}_mentioned' for name in CORE_FACILITIES]
destinations['facility_evidence_count'] = destinations[facility_columns].sum(axis=1)
destinations['c4_facility_score_heuristic'] = (
    destinations['facility_evidence_count'] / len(CORE_FACILITIES)
)
destinations['c4_facility_source'] = 'description_keyword_heuristic_unverified'
# Kolom ini dapat diganti dengan skor fasilitas terverifikasi tanpa mengubah pipeline selanjutnya.
destinations['c4_facility_score'] = destinations['c4_facility_score_heuristic']

display(destinations[['place_name', *facility_columns, 'c4_facility_score']].head(10))

## 4. Ekstraksi tag aktivitas untuk C6

Tag aktivitas diperoleh dari kategori dan kata kunci deskripsi. Hasilnya dipakai untuk menghitung Jaccard Similarity dengan hobi pengguna. Daftar aturan disimpan secara eksplisit agar dapat diperiksa dan diperbaiki oleh tim.

In [ ]:
ACTIVITY_KEYWORDS = {
    'hiking': ['hiking', 'mendaki', 'pendakian', 'trekking', 'jalur setapak'],
    'fotografi': ['fotografi', 'spot foto', 'berfoto', 'pemandangan', 'panorama', 'instagramable'],
    'snorkeling': ['snorkeling', 'snorkel'],
    'diving': ['diving', 'menyelam', 'selam'],
    'camping': ['camping', 'berkemah', 'bumi perkemahan'],
    'kuliner': ['kuliner', 'makanan khas', 'jajanan', 'warung', 'restoran'],
    'sejarah': ['sejarah', 'bersejarah', 'peninggalan', 'museum', 'monumen'],
    'budaya': ['budaya', 'tradisi', 'kesenian', 'keraton', 'cagar budaya'],
    'belanja': ['belanja', 'pusat perbelanjaan', 'pasar', 'mal', 'mall'],
    'berenang': ['berenang', 'kolam renang', 'waterpark', 'water park'],
    'edukasi': ['edukasi', 'pendidikan', 'belajar', 'museum', 'observatorium'],
    'religi': ['ziarah', 'religi', 'ibadah', 'masjid', 'gereja', 'pura', 'vihara'],
    'rekreasi_keluarga': ['keluarga', 'wahana', 'taman bermain', 'taman hiburan'],
    'surfing': ['surfing', 'berselancar', 'selancar'],
}
CATEGORY_SEED_TAGS = {
    'budaya': {'budaya'},
    'belanja': {'belanja'},
    'religi': {'religi'},
    'hiburan': {'rekreasi_keluarga'},
}

def derive_activity_tags(row: pd.Series) -> list[str]:
    text = normalize_for_search(f"{row.get('place_name', '')} {row.get('description', '')}")
    tags = set(CATEGORY_SEED_TAGS.get(str(row.get('category_clean', '')), set()))
    for tag, keywords in ACTIVITY_KEYWORDS.items():
        if any(contains_keyword(text, keyword) for keyword in keywords):
            tags.add(tag)
    return sorted(tags)

destinations['activity_tags_list'] = destinations.apply(derive_activity_tags, axis=1)
destinations['activity_tags'] = destinations['activity_tags_list'].map(lambda tags: '|'.join(tags))
destinations['activity_tag_count'] = destinations['activity_tags_list'].map(len)
destinations['activity_tags_source'] = 'description_and_category_heuristic_unverified'

display(destinations[['place_name', 'category_clean', 'activity_tags']].head(15))

## 5. Dataset siap K-Means

Fitur numerik distandardisasi agar harga tidak mendominasi rating dan fasilitas. Kategori diubah dengan one-hot encoding karena kategori tidak memiliki urutan alami. Jarak dan kecocokan personal tidak digunakan untuk clustering karena nilainya bergantung pada pengguna.

In [ ]:
cluster_source = destinations.loc[destinations['eligible_for_ranking']].copy()
CLUSTER_NUMERIC_FEATURES = ['c1_ticket_price', 'c2_rating', 'c4_facility_score']

cluster_numeric = cluster_source[CLUSTER_NUMERIC_FEATURES].astype(float)
feature_mean = cluster_numeric.mean()
feature_scale = cluster_numeric.std(ddof=0).replace(0, 1)
scaled_values = (cluster_numeric - feature_mean) / feature_scale
scaled_columns = [f'{column}_z' for column in CLUSTER_NUMERIC_FEATURES]
scaled_df = pd.DataFrame(scaled_values.to_numpy(), columns=scaled_columns, index=cluster_source.index)
category_dummies = pd.get_dummies(
    cluster_source['category_clean'], prefix='category', dtype=int
)

cluster_ready = pd.concat([
    cluster_source[['place_id', 'place_name', 'city', 'province', 'category_clean']].reset_index(drop=True),
    scaled_df.reset_index(drop=True),
    category_dummies.reset_index(drop=True),
], axis=1)

scaler_metadata = {
    'features': CLUSTER_NUMERIC_FEATURES,
    'mean': feature_mean.to_dict(),
    'scale': feature_scale.to_dict(),
    'note': 'C4 masih menggunakan skor heuristik sampai data fasilitas terverifikasi tersedia.',
}
display(cluster_ready.head())
print('Shape dataset K-Means:', cluster_ready.shape)

## 6. Fungsi dinamis C3, C5, dan C6

Ketiga nilai ini dihitung setelah jawaban survei pengguna tersedia. Tidak ada bobot AHP atau perhitungan TOPSIS di notebook preprocessing ini.

In [ ]:
def haversine_km(origin_lat, origin_lon, destination_lat, destination_lon):
    """Jarak garis besar bumi dalam kilometer; mendukung Series/array."""
    earth_radius_km = 6371.0088
    lat1 = np.radians(np.asarray(origin_lat, dtype=float))
    lon1 = np.radians(np.asarray(origin_lon, dtype=float))
    lat2 = np.radians(np.asarray(destination_lat, dtype=float))
    lon2 = np.radians(np.asarray(destination_lon, dtype=float))
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    a = np.sin(delta_lat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
    return earth_radius_km * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# Kosong secara sengaja. Tambahkan hubungan kategori hanya setelah disepakati dan divalidasi tim.
RELATED_CATEGORY_MAP: dict[str, set[str]] = {}

def category_match_score(destination_category: str, preferred_categories, related_map=None) -> float:
    preferred = {str(item).strip().casefold() for item in (preferred_categories or []) if str(item).strip()}
    destination_category = str(destination_category).strip().casefold()
    if destination_category in preferred:
        return 1.0
    related_map = related_map or {}
    if any(destination_category in related_map.get(category, set()) for category in preferred):
        return 0.5
    return 0.0

def parse_tags(value) -> set[str]:
    if isinstance(value, (list, set, tuple)):
        values = value
    elif pd.isna(value):
        values = []
    else:
        values = re.split(r'[|,;]', str(value))
    return {str(item).strip().casefold() for item in values if str(item).strip()}

def jaccard_similarity(left, right) -> float:
    left_set, right_set = parse_tags(left), parse_tags(right)
    union = left_set | right_set
    return len(left_set & right_set) / len(union) if union else 0.0

print('Contoh Jaccard:', jaccard_similarity(['hiking', 'fotografi'], ['hiking', 'camping']))

## 7. Membentuk matriks keputusan C1–C6

Format profil berikut dapat diisi dari hasil survei atau form aplikasi:

- `budget_max`: batas harga tiket.
- `origin_lat`, `origin_lon`: lokasi asal pengguna.
- `allowed_cities`: kota tujuan yang dipilih.
- `preferred_categories`: kategori minat.
- `hobbies`: hobi/aktivitas.
- `priority_profile`: profil AHP; dipakai pada notebook/model berikutnya.

In [ ]:
def build_decision_matrix(destination_df: pd.DataFrame, user_profile: dict) -> pd.DataFrame:
    required_profile_keys = {
        'budget_max', 'origin_lat', 'origin_lon',
        'preferred_categories', 'hobbies',
    }
    missing_keys = required_profile_keys - set(user_profile)
    if missing_keys:
        raise ValueError(f'Profil pengguna belum memiliki: {sorted(missing_keys)}')

    candidates = destination_df.loc[destination_df['eligible_for_ranking']].copy()
    candidates = candidates.loc[candidates['c1_ticket_price'] <= float(user_profile['budget_max'])]

    allowed_cities = {
        str(city).strip().casefold()
        for city in user_profile.get('allowed_cities', [])
        if str(city).strip()
    }
    if allowed_cities:
        candidates = candidates.loc[candidates['city'].str.casefold().isin(allowed_cities)]

    candidates['c3_distance_km'] = haversine_km(
        user_profile['origin_lat'], user_profile['origin_lon'],
        candidates['lat'], candidates['long'],
    )
    candidates['c5_category_match'] = candidates['category_clean'].map(
        lambda category: category_match_score(
            category, user_profile['preferred_categories'], RELATED_CATEGORY_MAP
        )
    )
    candidates['c6_hobby_match'] = candidates['activity_tags'].map(
        lambda tags: jaccard_similarity(user_profile['hobbies'], tags)
    )

    matrix_columns = [
        'place_id', 'place_name', 'city', 'province', 'category_clean', 'activity_tags',
        'c1_ticket_price', 'c2_rating', 'c3_distance_km',
        'c4_facility_score', 'c5_category_match', 'c6_hobby_match',
        'c4_facility_source', 'activity_tags_source',
    ]
    return candidates[matrix_columns].sort_values(['place_name']).reset_index(drop=True)

example_profile = {
    'respondent_id': 'CONTOH-001',
    'budget_max': 100_000,
    'origin_lat': -6.2000,
    'origin_lon': 106.8167,
    'allowed_cities': ['Jakarta'],
    'preferred_categories': ['budaya', 'alam'],
    'hobbies': ['fotografi', 'sejarah', 'kuliner'],
    'priority_profile': 'seimbang',
}

decision_matrix_example = build_decision_matrix(destinations, example_profile)
display(decision_matrix_example.head(10))
print('Jumlah kandidat setelah filter:', len(decision_matrix_example))

## 8. Laporan kualitas dan ekspor

Output utama:

- `destinations_clean.csv`: data destinasi bersih beserta provenance dan flag kualitas.
- `destinations_kmeans_ready.csv`: fitur numerik terstandardisasi dan kategori one-hot.
- `ratings_aggregated.csv`: statistik rating pengguna per destinasi.
- `decision_matrix_example.csv`: contoh input C1–C6 untuk AHP–TOPSIS.
- `preprocessing_quality_report.json`: ringkasan kualitas dan catatan metodologis.
- `scaler_metadata.json`: parameter StandardScaler agar transformasi dapat direproduksi.

In [ ]:
quality_report = {
    'raw_rows': {
        'destinations': int(len(destinations_raw)),
        'ratings': int(len(ratings_raw)),
    },
    'processed_rows': {
        'destinations': int(len(destinations)),
        'eligible_for_ranking': int(destinations['eligible_for_ranking'].sum()),
        'valid_ratings': int(len(ratings_valid)),
    },
    'quality_flags': {
        'duplicate_place_id': int(destinations['duplicate_place_id'].sum()),
        'duplicate_name_coordinate': int(destinations['duplicate_name_coordinate'].sum()),
        'invalid_price': int((~destinations['valid_price']).sum()),
        'invalid_rating': int((~destinations['valid_rating']).sum()),
        'invalid_coordinate': int((~destinations['valid_coordinate']).sum()),
        'outside_indonesia_bbox_warning': int((~destinations['inside_indonesia_bbox']).sum()),
        'missing_time_minutes': int(destinations['time_minutes_missing'].sum()),
    },
    'coverage': {
        'cities': sorted(destinations['city'].dropna().unique().tolist()),
        'provinces': sorted(destinations['province'].dropna().unique().tolist()),
        'categories': sorted(destinations['category_clean'].dropna().unique().tolist()),
        'destinations_with_facility_evidence': int(destinations['facility_evidence_count'].gt(0).sum()),
        'destinations_with_activity_tags': int(destinations['activity_tag_count'].gt(0).sum()),
    },
    'method_notes': [
        'C1 dan C2 berasal langsung dari dataset destinasi.',
        'C3 dihitung secara dinamis menggunakan Haversine.',
        'C4 sementara merupakan heuristik penyebutan fasilitas pada deskripsi.',
        'C5 exact match; related-category map sengaja kosong sampai divalidasi.',
        'C6 menggunakan Jaccard pada tag aktivitas hasil ekstraksi heuristik.',
        'Budget dan wilayah adalah filter keras; kategori dan hobi bukan filter keras.',
    ],
}

print(json.dumps(quality_report, ensure_ascii=False, indent=2))

In [ ]:
# List Python tidak disimpan langsung ke CSV; versi string activity_tags sudah tersedia.
destinations_export = destinations.drop(columns=['activity_tags_list']).copy()

output_files = {
    'destinations_clean': PROCESSED_DIR / 'destinations_clean.csv',
    'destinations_kmeans_ready': PROCESSED_DIR / 'destinations_kmeans_ready.csv',
    'ratings_aggregated': PROCESSED_DIR / 'ratings_aggregated.csv',
    'decision_matrix_example': PROCESSED_DIR / 'decision_matrix_example.csv',
    'quality_report': REPORT_DIR / 'preprocessing_quality_report.json',
    'scaler_metadata': REPORT_DIR / 'scaler_metadata.json',
}

destinations_export.to_csv(output_files['destinations_clean'], index=False, encoding='utf-8-sig')
cluster_ready.to_csv(output_files['destinations_kmeans_ready'], index=False, encoding='utf-8-sig')
rating_summary.to_csv(output_files['ratings_aggregated'], index=False, encoding='utf-8-sig')
decision_matrix_example.to_csv(output_files['decision_matrix_example'], index=False, encoding='utf-8-sig')

with open(output_files['quality_report'], 'w', encoding='utf-8') as file:
    json.dump(quality_report, file, ensure_ascii=False, indent=2)
with open(output_files['scaler_metadata'], 'w', encoding='utf-8') as file:
    json.dump(scaler_metadata, file, ensure_ascii=False, indent=2)

for label, path in output_files.items():
    print(f'{label:28s} -> {path.relative_to(PROJECT_ROOT)}')

In [ ]:
# Pemeriksaan akhir otomatis. Cell akan berhenti jika kontrak data utama tidak terpenuhi.
assert destinations['place_id'].notna().all(), 'Masih ada place_id kosong.'
assert destinations['place_id'].is_unique, 'place_id belum unik.'
assert destinations.loc[destinations['valid_rating'], 'c2_rating'].between(1, 5).all()
assert destinations.loc[destinations['valid_price'], 'c1_ticket_price'].ge(0).all()
assert destinations['c4_facility_score'].between(0, 1).all()
assert decision_matrix_example['c5_category_match'].between(0, 1).all()
assert decision_matrix_example['c6_hobby_match'].between(0, 1).all()
assert not cluster_ready.isna().any().any(), 'Dataset K-Means masih memiliki missing value.'
print('OK - Seluruh pemeriksaan akhir berhasil.')

## Langkah berikutnya

1. Verifikasi C4 dan tag aktivitas pada sampel destinasi.
2. Ganti `c4_facility_score` dengan skor fasilitas terverifikasi jika tersedia.
3. Impor hasil survei pengguna dan bentuk matriks C1–C6 untuk setiap responden/skenario.
4. Jalankan Elbow Method, Silhouette Score, dan K-Means menggunakan `destinations_kmeans_ready.csv`.
5. Hitung bobot AHP, pastikan CR < 0,1, lalu lakukan ranking TOPSIS.
6. Bandingkan TOPSIS dengan SAW serta lakukan sensitivity analysis dan uji rank reversal.